Exercise 1: Training a trigram model.

I will implement the trigram model using a neural network.

Unlike the bigram model, which just took in one previous character and predict the next character, the trigram model will take in two previous characters and predict the next character.

In [1]:
import gdown
gdown.download("https://drive.google.com/uc?id=1D3VgZXIVUCxv2dZSs7cGKtbu_VcPerSw", "names.txt", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1D3VgZXIVUCxv2dZSs7cGKtbu_VcPerSw
To: /content/names.txt
100%|██████████| 228k/228k [00:00<00:00, 67.4MB/s]


'names.txt'

In [2]:
names = open('names.txt', 'r').read().splitlines()

In [3]:
names[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [4]:
len(names)

32033

In [5]:
from string import ascii_lowercase

characters = list(ascii_lowercase)
characters = ['.'] + characters
characters

['.',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [6]:
# Defining the ctoi mapping -- this maps a character to it's corresponding index.

ctoi = {c: i for i, c in enumerate(characters)}
ctoi

{'.': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26}

In [7]:
len(ctoi)

27

In [8]:
# Defining the itoc mapping -- this maps an index to its corresponding character.
itoc = {i: c for c, i in ctoi.items()}
itoc

{0: '.',
 1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z'}

In [9]:
len(itoc)

27

In [10]:
# Defining the cptoi mapping -- this maps a character pair to it's corresponding index.

cptoi = {(c1, c2): i for i, (c1, c2) in enumerate((c1, c2) for c1 in characters for c2 in characters)}

In [11]:
len(cptoi)

729

Ex. for the first word in the training dataset ('emma'):

'emma' -> '..emma.'


Example 1: .. -> e

Example 2: .e -> m

Example 3: em -> m

Example 4: mm -> a

Example 5: ma -> .

In [12]:
import torch

# Creating the training set of trigrams.
xs, ys = [], []

for name in names:
  cs = ['.', '.'] + list(name) + ['.']
  for c1, c2, c3 in zip(cs, cs[1:], cs[2:]):
    xs.append(cptoi[(c1, c2)])
    ys.append(ctoi[c3])

xs, ys = torch.tensor(xs), torch.tensor(ys)


In [13]:
xs.shape

torch.Size([228146])

In [14]:
ys.shape

torch.Size([228146])

In [15]:
assert len(xs) == len(ys)

n = len(xs)

Previously, with the bigram neural network, we simply encoded each input character as a 27-length OHE vector that was fed as input to the neural network.

What do we do with a trigram neural network?

Well, one way to do this is to simply encode the pair of input characters as a 729-length OHE vector that we will feed as input to the neural network.


Right now, xs is of shape 228146, and I want it to become of shape 228146 x 729.


In [16]:
# One-Hot Encoding (OHE)

import torch.nn.functional as F

def one_hot_encoding(raw):
  return F.one_hot(raw, num_classes = 729).float()

xs_encoded = one_hot_encoding(xs)
xs_encoded.shape

torch.Size([228146, 729])

Note: Another way to do this would be to take the first character in the pair, run it through a one-hot encoder, get the 27-length OHE vector result. Then, take the second character in the pair, run it through a one-hot encoder, get the 27-length OHE vector result. Then, take the two resulting 27-length OHE vectors, and concatenate them together to form a 54-length vector. I can try this later.

In [27]:
# Now, let's check our work.

# The second training example is: .e -> m

# Therefore, at xs[1], we expect to see 5 (because cptoi[('.', 'e')] = 5)

xs[1].item()

5

In [31]:
# At xs_encoded[1], we expect to see a 729-length vector that has a 1 at index 5.

print(xs_encoded[1].argmax().item())

5


What will the trigram neural network look like?


Let's recall what the bigram neural network looked like.

It consisted of an input layer, which was of size 27 (recall that the input to the bigram neural network was a 27-length OHE).

And then just an output layer, which was of size 27, which outputted the probability distribution over the next character.


Extending this to the trigram neural network, we can similarly start with:

An input layer, which will be of size 729 (recall that the input to the trigram neural network is a 729-length OHE).

And then just an output layer, which will be of size 27, which will output the probability distribution over the next character.



In [ ]:
# Let's randomly generate the weights for every one of the 27 neurons in the layer.

g = torch.Generator().manual_seed(42)
W = torch.randn((729, 27), generator = g, requires_grad = True)

In [ ]:
# Defining the softmax function.

def softmax(logits):
  # As Andrej said, these two lines are considered to be the 'softmax'.
  counts = logits.exp()
  probabilities = counts / counts.sum(1, keepdims = True)
  return probabilities


In [ ]:
# Main training loop.

num_epochs = 10

for epoch in range(num_epochs):

  # Forward pass.
  logits = xs_encoded @ W
  probabilities = softmax(logits)
  loss = -probabilities[torch.arange(n), ys].log().mean()

  # Print loss.
  print(epoch, loss.item())

  # Backward pass.
  W.grad = None
  loss.backward()

  # Update.
  W.data -= 0.01 * W.grad

0 2.3044092655181885
1 2.3044092655181885
2 2.3044092655181885
3 2.3044090270996094
4 2.3044090270996094
5 2.3044090270996094
6 2.3044090270996094
7 2.3044090270996094
8 2.3044090270996094
9 2.3044090270996094


In [ ]:
"""

Let's consider again why this is correct for NLL:

loss = -probabilities[torch.arange(n), ys].log().mean()

probabilities is a n x 27 tensor-- there are n training inputs, and, for each of those inputs, it's giving us the model's predicted probability distribution over the 27 possible next characters.

probabilities[torch.arange(n), ys] is a way of indexing it that is the same as:

[probabilities[0, ys[0],
 probabilities[1, ys[1],
 probabilities[2, ys[2],
 ...
 probabilities[n - 1, ys[n - 1]]

This is just plucking out the probabilities the model is assigning to the correct characters...


...

"""

In [ ]:
# Loss with the trigram model is at ~2.30 and still dropping, which is better than the loss with Karpathy's bigram model (~2.46).

In [ ]:
# Sample from the trigram model (trivial)...

# Just make sure you seed it with ['.', '.'] as the first two characters.

Exercise 2: Splitting the dataset randomly into an 80/10/10 split.

Training dataset: 80% of the total names.

Development dataset: 10% of the total names.

Testing dataset: 10% of the total names.

In [ ]:
import random

In [ ]:
random.shuffle(names)

In [ ]:
print(f"Total number of names in the dataset: {len(names)}")

Total number of names in the dataset: 32033


In [ ]:
train_names = names[:int(0.8 * len(names))]
dev_names = names[int(0.8 * len(names)):int(0.9 * len(names))]
test_names = names[int(0.9 * len(names)):]

In [ ]:
print(f"Total number of names in the train dataset: {len(train_names)}")

Total number of names in the train dataset: 25626


In [ ]:
print(f"Total number of names in the dev dataset: {len(dev_names)}")

Total number of names in the dev dataset: 3203


In [ ]:
print(f"Total number of names in the test dataset: {len(test_names)}")

Total number of names in the test dataset: 3204


In [ ]:
# Then proceed to build the xs and ys for each split (trivial)...

Exercise 3: Playing with the regularization of the trigram model.

In [ ]:
# Skip (for now) (trivial)...

Exercise 4: Getting rid of one-hot encoding (OHE).

In [ ]:
# Skip (for now) (trivial)...

Exercise 5: Using cross-entropy loss instead.

What is F.cross_entropy?

Currently, in the forward pass, we are computing the softmax and then the NLL in two separate stages. This new function combines both so that we compute both of them in a single stage.




In [ ]:
# We can, thus, replace it with F.cross_entropy and we should get the same result (trivial)...